In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
nav_data = pd.read_csv("../data/raw/02_nav_history.csv")

In [3]:
nav_data.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [4]:
nav_data.columns

Index(['amfi_code', 'date', 'nav'], dtype='str')

In [5]:
nav_data['date'] = pd.to_datetime(nav_data['date'])

In [6]:
nav_data = nav_data.sort_values(['amfi_code', 'date'])

In [7]:
nav_data['daily_return'] = (
    nav_data.groupby('amfi_code')['nav']
            .pct_change()
)

In [8]:
returns = nav_data.dropna(subset=['daily_return'])

In [9]:
risk_metrics = []

for fund in returns['amfi_code'].unique():

    fund_returns = returns.loc[
        returns['amfi_code'] == fund,
        'daily_return'
    ]

    # 95% Historical VaR (5th percentile)
    var95 = np.percentile(fund_returns, 5)

    # CVaR (average return below VaR)
    cvar95 = fund_returns[fund_returns <= var95].mean()

    risk_metrics.append({
        "AMFI_Code": fund,
        "VaR_95": var95,
        "CVaR_95": cvar95
    })

risk_df = pd.DataFrame(risk_metrics)

In [10]:
risk_df.head()

,AMFI_Code,VaR_95,CVaR_95
0,100016,-0.014364,-0.018060
1,100025,-0.003793,-0.004994
2,100033,-0.019034,-0.023456
3,101206,-0.013282,-0.017439
4,101207,-0.026021,-0.032459


In [11]:
risk_df = risk_df.sort_values("VaR_95")
risk_df

,AMFI_Code,VaR_95,CVaR_95
22,119599,-0.026859,-0.032384
17,119095,-0.026188,-0.031667
4,101207,-0.026021,-0.032459
11,118634,-0.025438,-0.032304
21,119598,-0.024507,-0.030595
39,149324,-0.023483,-0.031036
7,102886,-0.019220,-0.023251
2,100033,-0.019034,-0.023456
25,120505,-0.018892,-0.024342
16,119094,-0.018480,-0.024260


In [12]:
risk_df.to_csv("var_cvar_report.csv", index=False)

In [13]:
returns.head()

,amfi_code,date,nav,daily_return
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639


In [14]:
returns['rolling_mean'] = (
    returns.groupby('amfi_code')['daily_return']
           .transform(lambda x: x.rolling(90).mean())
)

In [15]:
returns['rolling_std'] = (
    returns.groupby('amfi_code')['daily_return']
           .transform(lambda x: x.rolling(90).std())
)

In [16]:
import numpy as np

returns['rolling_sharpe'] = (
    returns['rolling_mean'] /
    returns['rolling_std']
) * np.sqrt(252)

In [17]:
returns.head()

,amfi_code,date,nav,daily_return,rolling_mean,rolling_std,rolling_sharpe
5751,100016,2022-01-04,515.0971,-0.010306,NaN,NaN,NaN
5752,100016,2022-01-05,521.7239,0.012865,NaN,NaN,NaN
5753,100016,2022-01-06,515.7880,-0.011377,NaN,NaN,NaN
5754,100016,2022-01-07,515.1639,-0.001210,NaN,NaN,NaN
5755,100016,2022-01-10,510.7136,-0.008639,NaN,NaN,NaN


In [18]:
returns['amfi_code'].unique()

array([100016, 100025, 100033, 101206, 101207, 101208, 102885, 102886,
       102887, 118632, 118633, 118634, 118635, 118636, 119092, 119093,
       119094, 119095, 119120, 119551, 119552, 119598, 119599, 120503,
       120504, 120505, 120506, 120507, 120841, 120842, 120843, 120844,
       125497, 125498, 148567, 148568, 148569, 149322, 149323, 149324])

In [19]:
top5_funds = returns['amfi_code'].unique()[:5]

In [20]:
plot_data = returns[
    returns['amfi_code'].isin(top5_funds)
]

In [21]:
import plotly.express as px

fig = px.line(
    plot_data,
    x='date',
    y='rolling_sharpe',
    color='amfi_code',
    title='Rolling 90-Day Sharpe Ratio (Top 5 Funds)'
)

fig.show()

In [22]:
fig.write_image("rolling_sharpe_chart.png")

In [25]:
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")

In [26]:
transactions.columns

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='str')

In [27]:
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date']
)

In [28]:
first_transaction = (
    transactions
    .groupby('investor_id')['transaction_date']
    .min()
    .reset_index()
)

In [29]:
first_transaction['cohort_year'] = (
    first_transaction['transaction_date'].dt.year
)

In [30]:
transactions = transactions.merge(
    first_transaction[['investor_id', 'cohort_year']],
    on='investor_id',
    how='left'
)

In [31]:
avg_sip = (
    transactions
    .groupby('cohort_year')['amount_inr']
    .mean()
    .reset_index(name='Average_SIP')
)

avg_sip

,cohort_year,Average_SIP
0,2024,107422.541832
1,2025,109158.577061


In [32]:
total_investment = (
    transactions
    .groupby('cohort_year')['amount_inr']
    .sum()
    .reset_index(name='Total_Investment')
)

total_investment

,cohort_year,Total_Investment
0,2024,3491125187
1,2025,30455243


In [33]:
top_fund = (
    transactions
    .groupby(['cohort_year', 'amfi_code'])
    .size()
    .reset_index(name='count')
)

In [34]:
top_fund = (
    top_fund
    .sort_values(['cohort_year', 'count'], ascending=[True, False])
    .drop_duplicates('cohort_year')
)

top_fund

,cohort_year,amfi_code,count
35,2024,148568,874
62,2025,119599,12


In [35]:
cohort_summary = (
    avg_sip
    .merge(total_investment, on='cohort_year')
    .merge(
        top_fund[['cohort_year', 'amfi_code']],
        on='cohort_year'
    )
)

cohort_summary

,cohort_year,Average_SIP,Total_Investment,amfi_code
0,2024,107422.541832,3491125187,148568
1,2025,109158.577061,30455243,119599


In [36]:
sip_data = transactions[
    transactions['transaction_type'] == 'SIP'
].copy()

In [37]:
sip_data = transactions.copy()

In [38]:
sip_data['transaction_date'] = pd.to_datetime(
    sip_data['transaction_date']
)

In [39]:
sip_data = sip_data.sort_values(
    ['investor_id', 'transaction_date']
)

In [40]:
sip_data['gap_days'] = (
    sip_data.groupby('investor_id')['transaction_date']
            .diff()
            .dt.days
)

In [41]:
sip_count = (
    sip_data.groupby('investor_id')
            .size()
            .reset_index(name='sip_count')
)

sip_count.head()

,investor_id,sip_count
0,INV000001,3
1,INV000002,6
2,INV000003,2
3,INV000004,9
4,INV000005,8


In [42]:
eligible_investors = sip_count[
    sip_count['sip_count'] >= 6
]['investor_id']

In [43]:
eligible_data = sip_data[
    sip_data['investor_id'].isin(eligible_investors)
]

In [44]:
avg_gap = (
    eligible_data.groupby('investor_id')['gap_days']
                 .mean()
                 .reset_index(name='avg_gap_days')
)

avg_gap.head()

,investor_id,avg_gap_days
0,INV000002,82.800000
1,INV000004,53.375000
2,INV000005,52.000000
3,INV000006,99.000000
4,INV000008,50.285714


In [45]:
avg_gap['status'] = avg_gap['avg_gap_days'].apply(
    lambda x: 'At-Risk' if x > 35 else 'Regular'
)

In [46]:
avg_gap.head(10)

,investor_id,avg_gap_days,status
0,INV000002,82.800000,At-Risk
1,INV000004,53.375000,At-Risk
2,INV000005,52.000000,At-Risk
3,INV000006,99.000000,At-Risk
4,INV000008,50.285714,At-Risk
5,INV000009,80.833333,At-Risk
6,INV000010,32.800000,Regular
7,INV000011,45.200000,At-Risk
8,INV000012,41.400000,At-Risk
9,INV000013,36.888889,At-Risk


In [47]:
avg_gap['status'].value_counts()

status
At-Risk    2762
Regular     188
Name: count, dtype: int64

In [48]:
import plotly.express as px

status_count = avg_gap['status'].value_counts().reset_index()
status_count.columns = ['Status', 'Count']

fig = px.bar(
    status_count,
    x='Status',
    y='Count',
    color='Status',
    title='SIP Continuity Analysis'
)

fig.show()

In [50]:
import pandas as pd

holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")

In [51]:
holdings.head()

,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,6011.08,2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,1074.65,2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,5964.59,2025-12-31
3,119551,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76,161.32,3748.82,2025-12-31
4,119551,ASIANPAINT,Asian Paints Ltd,Paints,10.25,725.90,1321.45,2025-12-31


In [52]:
holdings.columns

Index(['amfi_code', 'stock_symbol', 'stock_name', 'sector', 'weight_pct',
       'market_value_cr', 'current_price_inr', 'portfolio_date'],
      dtype='str')

In [53]:
holdings.head()


,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,6011.08,2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,1074.65,2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,5964.59,2025-12-31
3,119551,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76,161.32,3748.82,2025-12-31
4,119551,ASIANPAINT,Asian Paints Ltd,Paints,10.25,725.90,1321.45,2025-12-31


In [54]:
holdings['weight_decimal'] = holdings['weight_pct'] / 100

In [55]:
holdings['weight_square'] = holdings['weight_decimal'] ** 2

In [56]:
hhi = holdings.groupby('amfi_code')['weight_square'].sum().reset_index()

In [57]:
hhi.columns = ['amfi_code', 'HHI']

In [58]:
hhi.head()

,amfi_code,HHI
0,100016,0.139534
1,100033,0.147592
2,101206,0.129332
3,101207,0.200700
4,102885,0.174709


In [59]:
hhi = hhi.sort_values(
    by='HHI',
    ascending=False
)

hhi.head(10)


,amfi_code,HHI
11,119092,0.206448
3,101207,0.200700
18,119599,0.174751
4,102885,0.174709
7,118632,0.168298
29,148568,0.167930
21,120505,0.157570
22,120506,0.153794
27,125498,0.152414
23,120841,0.149680


In [60]:
import plotly.express as px

fig = px.bar(
    hhi.head(10),
    x='amfi_code',
    y='HHI',
    title='Top 10 Funds by Sector Concentration (HHI)',
    labels={
        'amfi_code': 'Fund (AMFI Code)',
        'HHI': 'Herfindahl-Hirschman Index'
    }
)

fig.show()

In [61]:
fig.write_image("sector_hhi_chart.png")

### Interpretation

- Higher HHI indicates that a fund is more concentrated in a few sectors.
- Lower HHI indicates that a fund is better diversified across multiple sectors.
- Highly concentrated funds may offer higher returns but also carry higher risk.

## Insight 1: Historical VaR & CVaR

Funds with the most negative VaR (95%) have the highest downside risk. These funds are more likely to experience larger daily losses during unfavorable market conditions.

**Reference:** VaR & CVaR Table (Task 1)

## Insight 2: Rolling Sharpe Ratio

Funds with a consistently higher Rolling 90-Day Sharpe Ratio delivered better risk-adjusted returns compared to other funds over time.

**Reference:** Rolling Sharpe Ratio Chart (Task 2)

## Insight 3: Investor Cohort Analysis

The investor cohort with the highest total investment showed stronger participation in mutual funds and preferred specific fund categories.

**Reference:** Cohort Analysis Table (Task 3)

## Insight 4: SIP Continuity

Most investors maintained regular SIP intervals, while investors with an average gap of more than 35 days were identified as "At-Risk".

**Reference:** SIP Continuity Analysis (Task 4)

## Insight 5: Sector HHI Concentration

Funds with higher HHI values are more concentrated in a few sectors, whereas lower HHI values indicate better diversification across sectors.

**Reference:** Sector HHI Bar Chart (Task 6)